# 07 · Decorators

A **decorator** is a function that *wraps* another function to add behavior —
without changing the wrapped function's code. It builds directly on closures
(the functions notebook). In data engineering you'll use decorators constantly: `@retry`,
timing, caching, logging, and framework hooks like `@pytest.fixture` or
pydantic's `@field_validator`.

## Functions are objects

The key idea that makes decorators possible: a function can be passed to another
function, returned, and assigned to a variable — just like any value.

In [ ]:
def shout(text):
    return text.upper() + '!'

f = shout               # assign the function to another name
print(f('etl'))

def apply(fn, value):   # pass a function as an argument
    return fn(value)
print(apply(shout, 'load'))

## Writing a decorator

A decorator takes a function and returns a new function that calls the original
with extra behavior around it. The `@decorator` line above a `def` is just
sugar for `func = decorator(func)`.

In [ ]:
def announce(fn):
    def wrapper(*args, **kwargs):
        print(f'-> calling {fn.__name__}')
        result = fn(*args, **kwargs)
        print(f'<- {fn.__name__} returned {result!r}')
        return result
    return wrapper

@announce                       # same as: add = announce(add)
def add(a, b):
    return a + b

add(3, 4)

## Preserve metadata with `functools.wraps`

The wrapper replaces the original, so `__name__` and the docstring get lost.
`@functools.wraps(fn)` copies them across — always use it, or debugging and
tooling break.

In [ ]:
import functools

def announce(fn):
    @functools.wraps(fn)        # copy fn's name/docstring onto wrapper
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper

@announce
def add(a, b):
    'Add two numbers.'
    return a + b

print(add.__name__)      # 'add', not 'wrapper'
print(add.__doc__)

## A practical decorator: timing

A `@timed` decorator measures how long any function takes — drop it on a
pipeline stage to profile it with zero code changes.

In [ ]:
import functools, time

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            ms = (time.perf_counter() - start) * 1000
            print(f'{fn.__name__} took {ms:.1f} ms')
    return wrapper

@timed
def crunch(n):
    return sum(i * i for i in range(n))

print('result:', crunch(500_000))

## Decorators that take arguments

To configure a decorator (e.g. how many times to retry), add one more layer: a
function that takes the arguments and *returns* the decorator. Here's a real
`@retry` — the single most common decorator in ingestion code.

In [ ]:
import functools, time

def retry(times=3, delay=0.01):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    print(f'attempt {attempt} failed: {e}')
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

_calls = {'n': 0}

@retry(times=3)
def flaky():
    _calls['n'] += 1
    if _calls['n'] < 3:
        raise ConnectionError('transient')
    return 'ok'

print('final:', flaky())

## `functools.lru_cache` — a built-in decorator

The standard library ships useful decorators. `@lru_cache` memoizes results so
repeated calls with the same arguments are instant — handy for expensive lookups.

In [ ]:
import functools

@functools.lru_cache(maxsize=None)
def slow_square(n):
    print(f'  computing {n}...')
    return n * n

print(slow_square(12))   # computes
print(slow_square(12))   # cached — no 'computing' line
print(slow_square(5))    # computes

### Recap

A decorator wraps a function to add behavior via a closure; `@name` means
`func = name(func)`; use `functools.wraps` to keep the wrapped function's
identity; add an outer layer for decorators with arguments (`@retry(times=3)`);
`functools.lru_cache` is a ready-made memoizer. Next: iterators and generators.